In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130
!pip install matplotlib
!pip install seaborn
!pip install torchmetrics
!pip install tqdm
!pip install librosa

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torchmetrics import ConfusionMatrix
import seaborn as sb
from torch.optim import Adam
from torch.optim import lr_scheduler
from torchmetrics.classification import MulticlassAccuracy
from tqdm import tqdm
import numpy as np

# Set seed for reproducibility
seed = 42
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

# Helper Functions

In [ ]:
# Unified training function that tracks loss and accuracy per epoch
def train_model(model, train_loader, val_loader, optimizer, loss_fn, epochs, scheduler=None, model_name="Model"):
    """Train a model and return per-epoch training/validation loss and accuracy."""
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        # --- Training ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        loop = tqdm(train_loader, desc=f"{model_name} Epoch [{epoch+1}/{epochs}]")
        for imgs, labels in loop:
            imgs, labels = imgs.cuda(), labels.cuda()
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            loop.set_postfix(loss=loss.item())

        train_loss = running_loss / total
        train_acc = correct / total
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)

        # --- Validation ---
        val_loss, val_acc = evaluate_model(model, val_loader, loss_fn)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if scheduler:
            scheduler.step()

        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    return history


def evaluate_model(model, loader, loss_fn=None):
    """Evaluate model on a dataloader. Returns (loss, accuracy)."""
    if loss_fn is None:
        loss_fn = nn.CrossEntropyLoss()
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.cuda(), labels.cuda()
            outputs = model(imgs)
            loss = loss_fn(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total


def plot_history(history, model_name="Model"):
    """Plot training and validation loss/accuracy curves."""
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history["train_loss"], label="Train Loss")
    ax1.plot(epochs, history["val_loss"], label="Val Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title(f"{model_name} - Loss")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs, history["train_acc"], label="Train Acc")
    ax2.plot(epochs, history["val_acc"], label="Val Acc")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title(f"{model_name} - Accuracy")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


def test_and_confusion_matrix(model, test_loader, class_names, model_name="Model"):
    """Evaluate on test set and plot confusion matrix. Returns test accuracy."""
    model.eval()
    conmat = ConfusionMatrix(task='multiclass', num_classes=10).cuda()
    test_accuracy = MulticlassAccuracy(num_classes=10).cuda()

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.cuda(), labels.cuda()
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            test_accuracy.update(predicted, labels)
            conmat.update(predicted, labels)

    acc = test_accuracy.compute().item()
    cm = conmat.compute().cpu().numpy()

    print(f"{model_name} - Test Accuracy: {acc:.4f}")

    plt.figure(figsize=(10, 7))
    sb.heatmap(cm, xticklabels=class_names, yticklabels=class_names, annot=True, fmt=".0f", cmap="Blues")
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    return acc

# Dataset for Models 1-4 (MEL Spectrogram Images)

In [ ]:
# Load the dataset using torchvision.datasets.ImageFolder (without transforms for visualisation)
image_dataset_raw = torchvision.datasets.ImageFolder("./GTZAN Dataset/images_original")
class_names = image_dataset_raw.classes

# Print name of classes present in the image folder
print(f"Classes: {class_names}")
print(f"Total images: {len(image_dataset_raw)}")

In [ ]:
# Visualising sample images from the dataset
fig = plt.figure(figsize=(15, 10))
fig.tight_layout()
plt.subplots_adjust(wspace=.1, hspace=.3)
for i in range(20):
    img, label = image_dataset_raw[i * 50]
    t = fig.add_subplot(4, 5, i + 1)
    t.set_title(f"{class_names[label]}")
    t.axes.get_xaxis().set_visible(False)
    t.axes.get_yaxis().set_visible(False)
    plt.imshow(img)
plt.show()

In [ ]:
# Load the dataset with transforms: resize to 180x180 and convert to tensor
image_transforms = T.Compose([
    T.Resize((180, 180)),
    T.ToTensor(),
])

image_dataset = torchvision.datasets.ImageFolder("./GTZAN Dataset/images_original", transform=image_transforms)

In [ ]:
# Train (70%), validation (20%) and test (10%) split
train_image_dataset, validation_image_dataset, test_image_dataset = torch.utils.data.random_split(
    image_dataset, [0.7, 0.2, 0.1], generator=torch.Generator().manual_seed(seed)
)
print(f"Train: {len(train_image_dataset)}, Val: {len(validation_image_dataset)}, Test: {len(test_image_dataset)}")

In [ ]:
# Loading the dataset into dataloaders
batch_size = 32
train_loader = DataLoader(train_image_dataset, batch_size=batch_size, num_workers=2, shuffle=True)
validation_loader = DataLoader(validation_image_dataset, batch_size=batch_size, num_workers=2, shuffle=False)
test_loader = DataLoader(test_image_dataset, batch_size=batch_size, num_workers=2, shuffle=False)

# Model 1 : Fully Connected Network with Two Hidden Layers

In [ ]:
Net1_input_size = 3 * 180 * 180
Net1_hidden_size1 = 512
Net1_hidden_size2 = 256
Net1_output_size = 10

class Net1(nn.Module):
    def __init__(self):
        super(Net1, self).__init__()
        self.fc1 = nn.Linear(Net1_input_size, Net1_hidden_size1)
        self.fc2 = nn.Linear(Net1_hidden_size1, Net1_hidden_size2)
        self.fc3 = nn.Linear(Net1_hidden_size2, Net1_output_size)

    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

## Net1 - 50 Epochs

In [ ]:
# Net1 training - 50 epochs
Net1_model_50 = Net1().cuda()
Net1_optimizer_50 = optim.SGD(Net1_model_50.parameters(), lr=0.01)
Net1_loss_fn = nn.CrossEntropyLoss()
Net1_scheduler_50 = lr_scheduler.StepLR(Net1_optimizer_50, step_size=30, gamma=0.5)

Net1_history_50 = train_model(Net1_model_50, train_loader, validation_loader,
                              Net1_optimizer_50, Net1_loss_fn, epochs=50,
                              scheduler=Net1_scheduler_50, model_name="Net1 (50 epochs)")

In [ ]:
plot_history(Net1_history_50, "Net1 (50 epochs)")

In [ ]:
Net1_acc_50 = test_and_confusion_matrix(Net1_model_50, test_loader, class_names, "Net1 (50 epochs)")

## Net1 - 100 Epochs

In [ ]:
# Net1 training - 100 epochs
Net1_model_100 = Net1().cuda()
Net1_optimizer_100 = optim.SGD(Net1_model_100.parameters(), lr=0.01)
Net1_scheduler_100 = lr_scheduler.StepLR(Net1_optimizer_100, step_size=30, gamma=0.5)

Net1_history_100 = train_model(Net1_model_100, train_loader, validation_loader,
                               Net1_optimizer_100, Net1_loss_fn, epochs=100,
                               scheduler=Net1_scheduler_100, model_name="Net1 (100 epochs)")

In [ ]:
plot_history(Net1_history_100, "Net1 (100 epochs)")

In [ ]:
Net1_acc_100 = test_and_confusion_matrix(Net1_model_100, test_loader, class_names, "Net1 (100 epochs)")

# Model 2 : Convolutional Network (Figure 1)

In [ ]:
# Architecture: conv1 -> conv2 -> MaxPool -> conv3 -> conv4 -> MaxPool -> FC+ReLU -> FC
# in_features for fc1 is computed from the output size after convolutions and pooling on 180x180 input
# conv1(3->32, k=7): 180-7+1=174 -> conv2(32->64, k=3): 174-3+1=172 -> pool(2): 86
# conv3(64->128, k=3): 86-3+1=84 -> conv4(128->128, k=3): 84-3+1=82 -> pool(2): 41
# Flatten: 128 * 41 * 41 = 215168

class Net2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2))

        self.flatten = nn.Flatten()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(in_features=128 * 41 * 41, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=10)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool1(x)

        x = self.relu(self.conv3(x))
        x = self.relu(self.conv4(x))
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Net2 - 50 Epochs

In [ ]:
Net2_model_50 = Net2().cuda()
Net2_optimizer_50 = Adam(Net2_model_50.parameters(), lr=0.001)
Net2_loss_fn = nn.CrossEntropyLoss()
Net2_scheduler_50 = lr_scheduler.StepLR(Net2_optimizer_50, step_size=30, gamma=0.5)

Net2_history_50 = train_model(Net2_model_50, train_loader, validation_loader,
                              Net2_optimizer_50, Net2_loss_fn, epochs=50,
                              scheduler=Net2_scheduler_50, model_name="Net2 (50 epochs)")

In [ ]:
plot_history(Net2_history_50, "Net2 (50 epochs)")

In [ ]:
Net2_acc_50 = test_and_confusion_matrix(Net2_model_50, test_loader, class_names, "Net2 (50 epochs)")

## Net2 - 100 Epochs

In [ ]:
Net2_model_100 = Net2().cuda()
Net2_optimizer_100 = Adam(Net2_model_100.parameters(), lr=0.001)
Net2_loss_fn = nn.CrossEntropyLoss()
Net2_scheduler_100 = lr_scheduler.StepLR(Net2_optimizer_100, step_size=30, gamma=0.5)

Net2_history_100 = train_model(Net2_model_100, train_loader, validation_loader,
                               Net2_optimizer_100, Net2_loss_fn, epochs=100,
                               scheduler=Net2_scheduler_100, model_name="Net2 (100 epochs)")

In [ ]:
plot_history(Net2_history_100, "Net2 (100 epochs)")

In [ ]:
Net2_acc_100 = test_and_confusion_matrix(Net2_model_100, test_loader, class_names, "Net2 (100 epochs)")

# Model 3 : Model 2 with Batch Normalisation Layer

In [ ]:
# Same architecture as Net2 but with BatchNorm after each convolution (before ReLU)
class Net3(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
        self.norm1 = nn.BatchNorm2d(num_features=32)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        self.norm2 = nn.BatchNorm2d(num_features=64)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2))

        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3)
        self.norm3 = nn.BatchNorm2d(num_features=128)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3)
        self.norm4 = nn.BatchNorm2d(num_features=128)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2))

        self.flatten = nn.Flatten()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(in_features=128 * 41 * 41, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=10)

    def forward(self, x):
        x = self.relu(self.norm1(self.conv1(x)))
        x = self.relu(self.norm2(self.conv2(x)))
        x = self.pool1(x)

        x = self.relu(self.norm3(self.conv3(x)))
        x = self.relu(self.norm4(self.conv4(x)))
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## Net3 - 50 Epochs

In [ ]:
Net3_model_50 = Net3().cuda()
Net3_optimizer_50 = Adam(Net3_model_50.parameters(), lr=0.0001)
Net3_loss_fn = nn.CrossEntropyLoss()
Net3_scheduler_50 = lr_scheduler.StepLR(Net3_optimizer_50, step_size=30, gamma=0.5)

Net3_history_50 = train_model(Net3_model_50, train_loader, validation_loader,
                              Net3_optimizer_50, Net3_loss_fn, epochs=50,
                              scheduler=Net3_scheduler_50, model_name="Net3 (50 epochs)")

In [ ]:
plot_history(Net3_history_50, "Net3 (50 epochs)")

In [ ]:
Net3_acc_50 = test_and_confusion_matrix(Net3_model_50, test_loader, class_names, "Net3 (50 epochs)")

## Net3 - 100 Epochs

In [ ]:
Net3_model_100 = Net3().cuda()
Net3_optimizer_100 = Adam(Net3_model_100.parameters(), lr=0.0001)
Net3_scheduler_100 = lr_scheduler.StepLR(Net3_optimizer_100, step_size=30, gamma=0.5)

Net3_history_100 = train_model(Net3_model_100, train_loader, validation_loader,
                               Net3_optimizer_100, Net3_loss_fn, epochs=100,
                               scheduler=Net3_scheduler_100, model_name="Net3 (100 epochs)")

In [ ]:
plot_history(Net3_history_100, "Net3 (100 epochs)")

In [ ]:
Net3_acc_100 = test_and_confusion_matrix(Net3_model_100, test_loader, class_names, "Net3 (100 epochs)")

# Model 4 : Model 3 with RMSProp Optimiser

## Net4 - 50 Epochs

In [ ]:
Net4_model_50 = Net3().cuda()
Net4_optimizer_50 = torch.optim.RMSprop(Net4_model_50.parameters(), lr=0.0003)
Net4_loss_fn = nn.CrossEntropyLoss()
Net4_scheduler_50 = lr_scheduler.StepLR(Net4_optimizer_50, step_size=30, gamma=0.5)

Net4_history_50 = train_model(Net4_model_50, train_loader, validation_loader,
                              Net4_optimizer_50, Net4_loss_fn, epochs=50,
                              scheduler=Net4_scheduler_50, model_name="Net4 (50 epochs)")

In [ ]:
plot_history(Net4_history_50, "Net4 (50 epochs)")

In [ ]:
Net4_acc_50 = test_and_confusion_matrix(Net4_model_50, test_loader, class_names, "Net4 (50 epochs)")

## Net4 - 100 Epochs

In [ ]:
Net4_model_100 = Net3().cuda()
Net4_optimizer_100 = torch.optim.RMSprop(Net4_model_100.parameters(), lr=0.0005)
Net4_loss_fn = nn.CrossEntropyLoss()
Net4_scheduler_100 = lr_scheduler.StepLR(Net4_optimizer_100, step_size=50, gamma=0.5)

Net4_history_100 = train_model(Net4_model_100, train_loader, validation_loader,
                               Net4_optimizer_100, Net4_loss_fn, epochs=100,
                               scheduler=Net4_scheduler_100, model_name="Net4 (100 epochs)")

In [ ]:
plot_history(Net4_history_100, "Net4 (100 epochs)")

In [ ]:
Net4_acc_100 = test_and_confusion_matrix(Net4_model_100, test_loader, class_names, "Net4 (100 epochs)")

# Model 5 : RNN with LSTMs (Audio Samples)

## Dataset for Models 5-6 (Audio Samples)

In [ ]:
import librosa
import numpy as np
import os
from tqdm import tqdm as tqdm_bar

class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, target_sr=22050, n_mels=64, max_len=130):
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.max_len = max_len
        self.n_mels = n_mels

        n_fft = 1024
        hop_length = 512
        corrupt_files = {"jazz.00054.wav"}

        file_paths = []
        labels = []
        for genre in self.classes:
            genre_path = os.path.join(root_dir, genre)
            if not os.path.isdir(genre_path):
                continue
            for fname in sorted(os.listdir(genre_path)):
                if fname.endswith(".wav") and fname not in corrupt_files:
                    file_paths.append(os.path.join(genre_path, fname))
                    labels.append(self.class_to_idx[genre])

        # Pre-compute all spectrograms once at load time
        self.data = []
        self.labels = []
        print(f"Loading {len(file_paths)} audio files...")
        for filepath, label in tqdm_bar(zip(file_paths, labels), total=len(file_paths)):
            waveform, _ = librosa.load(filepath, sr=target_sr, mono=True)
            mel = librosa.feature.melspectrogram(
                y=waveform, sr=target_sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
            )
            mel = np.log(mel + 1e-9)
            mel = (mel - mel.mean()) / (mel.std() + 1e-9)
            mel = torch.from_numpy(mel).float().transpose(0, 1)  # (time, n_mels)
            if mel.shape[0] > max_len:
                mel = mel[:max_len, :]
            elif mel.shape[0] < max_len:
                mel = torch.cat([mel, torch.zeros(max_len - mel.shape[0], n_mels)], dim=0)
            self.data.append(mel)
            self.labels.append(label)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Load audio dataset (spectrograms computed once here, then GPU training is not bottlenecked)
audio_root = "./GTZAN Dataset/genres_original"
audio_dataset = AudioDataset(audio_root, target_sr=22050, n_mels=64, max_len=130)
print(f"Total audio samples: {len(audio_dataset)}")
print(f"Classes: {audio_dataset.classes}")
print(f"Sample shape: {audio_dataset[0][0].shape}")

In [ ]:
# 70/20/10 split for audio dataset
audio_train, audio_val, audio_test = torch.utils.data.random_split(
    audio_dataset, [0.7, 0.2, 0.1], generator=torch.Generator().manual_seed(seed)
)

audio_batch_size = 32
audio_train_loader = DataLoader(audio_train, batch_size=audio_batch_size, shuffle=True, num_workers=0)
audio_val_loader = DataLoader(audio_val, batch_size=audio_batch_size, shuffle=False, num_workers=0)
audio_test_loader = DataLoader(audio_test, batch_size=audio_batch_size, shuffle=False, num_workers=0)

print(f"Train: {len(audio_train)}, Val: {len(audio_val)}, Test: {len(audio_test)}")

In [ ]:
class Net5(nn.Module):
    def __init__(self, input_size=64, hidden_size=256, num_layers=2, num_classes=10):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        # Mean-pool over all time steps instead of just the last hidden state
        out = out.mean(dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

In [ ]:
def train_audio_model(model, train_loader, val_loader, optimizer, loss_fn, max_epochs,
                      scheduler=None, early_stopping=None, model_name="Model"):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(max_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        loop = tqdm(train_loader, desc=f"{model_name} Epoch [{epoch+1}/{max_epochs}]")
        for data, labels in loop:
            data, labels = data.cuda(), labels.cuda()
            outputs = model(data)
            loss = loss_fn(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * data.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            loop.set_postfix(loss=loss.item())

        train_loss = running_loss / total
        train_acc = correct / total
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)

        val_loss, val_acc = evaluate_model(model, val_loader, loss_fn)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if scheduler:
            scheduler.step()

        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if early_stopping and early_stopping(val_loss):
            print(f"  Early stopping triggered at epoch {epoch+1}")
            break

    return history

In [ ]:
Net5_model = Net5(input_size=64, hidden_size=256, num_layers=2, num_classes=10).cuda()
Net5_optimizer = Adam(Net5_model.parameters(), lr=0.001)
Net5_scheduler = lr_scheduler.StepLR(Net5_optimizer, step_size=20, gamma=0.5)
Net5_loss_fn = nn.CrossEntropyLoss()
Net5_early_stopping = EarlyStopping(patience=15, min_delta=0.0005)

Net5_history = train_audio_model(Net5_model, audio_train_loader, audio_val_loader,
                                 Net5_optimizer, Net5_loss_fn, max_epochs=200,
                                 scheduler=Net5_scheduler,
                                 early_stopping=Net5_early_stopping, model_name="Net5")

In [ ]:
plot_history(Net5_history, "Net5 (LSTM)")

In [ ]:
Net5_acc = test_and_confusion_matrix(Net5_model, audio_test_loader, audio_dataset.classes, "Net5 (LSTM)")

# Model 6 : LSTM + GAN Augmentation (Audio Samples)

In [ ]:
# Conditional GAN for generating audio mel spectrogram features
# Generator: noise z + label -> mel spectrogram (seq_len, n_mels)
# Discriminator: mel spectrogram + label -> real/fake
# Uses one-hot encoding for class labels and ReLU activation

class Generator(nn.Module):
    def __init__(self, noise_dim=100, num_classes=10, seq_len=130, n_mels=64):
        super().__init__()
        self.seq_len = seq_len
        self.n_mels = n_mels
        self.num_classes = num_classes

        self.model = nn.Sequential(
            nn.Linear(noise_dim + num_classes, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.BatchNorm1d(1024),
            nn.Linear(1024, seq_len * n_mels),
            nn.Tanh()
        )

    def forward(self, noise, labels):
        # One-hot encode labels
        label_onehot = torch.zeros(labels.size(0), self.num_classes, device=labels.device)
        label_onehot.scatter_(1, labels.unsqueeze(1), 1)
        gen_input = torch.cat((noise, label_onehot), dim=1)
        out = self.model(gen_input)
        return out.view(-1, self.seq_len, self.n_mels)


class Discriminator(nn.Module):
    def __init__(self, num_classes=10, seq_len=130, n_mels=64):
        super().__init__()
        self.seq_len = seq_len
        self.n_mels = n_mels
        self.num_classes = num_classes

        self.model = nn.Sequential(
            nn.Linear(seq_len * n_mels + num_classes, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, data, labels):
        data_flat = data.view(-1, self.seq_len * self.n_mels)
        # One-hot encode labels
        label_onehot = torch.zeros(labels.size(0), self.num_classes, device=labels.device)
        label_onehot.scatter_(1, labels.unsqueeze(1), 1)
        disc_input = torch.cat((data_flat, label_onehot), dim=1)
        return self.model(disc_input)

In [ ]:
# Train the Conditional GAN on training audio mel spectrograms
noise_dim = 100
gan_epochs = 100
gan_lr = 0.0002

generator = Generator(noise_dim=noise_dim).cuda()
discriminator = Discriminator().cuda()

g_optimizer = Adam(generator.parameters(), lr=gan_lr)
d_optimizer = Adam(discriminator.parameters(), lr=gan_lr)

adversarial_loss = nn.BCELoss()

gan_g_losses = []
gan_d_losses = []

for epoch in range(gan_epochs):
    g_loss_epoch = 0
    d_loss_epoch = 0
    count = 0
    loop = tqdm(audio_train_loader, desc=f"GAN Epoch [{epoch+1}/{gan_epochs}]")
    for real_data, labels in loop:
        batch_size_cur = real_data.size(0)
        real_data, labels = real_data.cuda(), labels.cuda()

        # Labels for real and fake
        real_labels = torch.ones(batch_size_cur, 1).cuda()
        fake_labels = torch.zeros(batch_size_cur, 1).cuda()

        # --- Train Discriminator ---
        d_optimizer.zero_grad()

        # Real data
        real_output = discriminator(real_data, labels)
        d_loss_real = adversarial_loss(real_output, real_labels)

        # Fake data
        noise = torch.randn(batch_size_cur, noise_dim).cuda()
        fake_data = generator(noise, labels)
        fake_output = discriminator(fake_data.detach(), labels)
        d_loss_fake = adversarial_loss(fake_output, fake_labels)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        d_optimizer.step()

        # --- Train Generator ---
        g_optimizer.zero_grad()

        noise = torch.randn(batch_size_cur, noise_dim).cuda()
        fake_data = generator(noise, labels)
        fake_output = discriminator(fake_data, labels)
        g_loss = adversarial_loss(fake_output, real_labels)

        g_loss.backward()
        g_optimizer.step()

        g_loss_epoch += g_loss.item()
        d_loss_epoch += d_loss.item()
        count += 1
        loop.set_postfix(g_loss=g_loss.item(), d_loss=d_loss.item())

    gan_g_losses.append(g_loss_epoch / count)
    gan_d_losses.append(d_loss_epoch / count)

    print(f"  G Loss: {g_loss_epoch/count:.4f} | D Loss: {d_loss_epoch/count:.4f}")


In [ ]:
# Plot GAN training losses
plt.figure(figsize=(10, 5))
plt.plot(range(1, gan_epochs+1), gan_g_losses, label="Generator Loss")
plt.plot(range(1, gan_epochs+1), gan_d_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GAN Training Losses")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Generate augmented audio samples (same number as original training samples)
generator.eval()
generated_data = []
generated_labels = []

num_train = len(audio_train)
num_classes = 10
samples_per_class = num_train // num_classes

with torch.no_grad():
    for class_idx in range(num_classes):
        remaining = samples_per_class
        while remaining > 0:
            batch = min(remaining, 32)
            noise = torch.randn(batch, noise_dim).cuda()
            class_labels = torch.full((batch,), class_idx, dtype=torch.long).cuda()
            fake = generator(noise, class_labels)
            generated_data.append(fake.cpu())
            generated_labels.append(class_labels.cpu())
            remaining -= batch

    # Handle any remaining samples due to integer division
    extra = num_train - samples_per_class * num_classes
    if extra > 0:
        noise = torch.randn(extra, noise_dim).cuda()
        class_labels = torch.randint(0, num_classes, (extra,)).cuda()
        fake = generator(noise, class_labels)
        generated_data.append(fake.cpu())
        generated_labels.append(class_labels.cpu())

generated_data = torch.cat(generated_data, dim=0)
generated_labels = torch.cat(generated_labels, dim=0)

print(f"Generated {generated_data.shape[0]} augmented samples")
print(f"Generated data shape: {generated_data.shape}")

In [ ]:
# Combine original training data with GAN-generated data
class AugmentedAudioDataset(torch.utils.data.Dataset):
    def __init__(self, original_dataset, gen_data, gen_labels):
        self.original_dataset = original_dataset
        self.gen_data = gen_data
        self.gen_labels = gen_labels
        self.original_len = len(original_dataset)
        self.gen_len = len(gen_data)

    def __len__(self):
        return self.original_len + self.gen_len

    def __getitem__(self, idx):
        if idx < self.original_len:
            return self.original_dataset[idx]
        else:
            gen_idx = idx - self.original_len
            return self.gen_data[gen_idx], self.gen_labels[gen_idx].item()

augmented_dataset = AugmentedAudioDataset(audio_train, generated_data, generated_labels)
augmented_train_loader = DataLoader(augmented_dataset, batch_size=32, shuffle=True, num_workers=0)

print(f"Original training samples: {len(audio_train)}")
print(f"Augmented training samples: {len(augmented_dataset)}")

In [ ]:
# Net6 uses the same LSTM architecture as Net5
class Net6(nn.Module):
    def __init__(self, input_size=64, hidden_size=256, num_layers=2, num_classes=10):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out.mean(dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [ ]:
Net6_model = Net6(input_size=64, hidden_size=256, num_layers=2, num_classes=10).cuda()
Net6_optimizer = Adam(Net6_model.parameters(), lr=0.001)
Net6_scheduler = lr_scheduler.StepLR(Net6_optimizer, step_size=20, gamma=0.5)
Net6_loss_fn = nn.CrossEntropyLoss()
Net6_early_stopping = EarlyStopping(patience=15, min_delta=0.0005)

Net6_history = train_audio_model(Net6_model, augmented_train_loader, audio_val_loader,
                                 Net6_optimizer, Net6_loss_fn, max_epochs=200,
                                 scheduler=Net6_scheduler,
                                 early_stopping=Net6_early_stopping, model_name="Net6")

In [ ]:
plot_history(Net6_history, "Net6 (LSTM + GAN)")

In [ ]:
Net6_acc = test_and_confusion_matrix(Net6_model, audio_test_loader, audio_dataset.classes, "Net6 (LSTM + GAN)")

# Results Comparison

In [ ]:
# Summary table of all model test accuracies
results = {
    "Net1 (50 ep)": Net1_acc_50,
    "Net1 (100 ep)": Net1_acc_100,
    "Net2 (50 ep)": Net2_acc_50,
    "Net2 (100 ep)": Net2_acc_100,
    "Net3 (50 ep)": Net3_acc_50,
    "Net3 (100 ep)": Net3_acc_100,
    "Net4 (50 ep)": Net4_acc_50,
    "Net4 (100 ep)": Net4_acc_100,
    "Net5 (LSTM)": Net5_acc,
    "Net6 (LSTM+GAN)": Net6_acc,
}

print("=" * 50)
print(f"{'Model':<35} {'Test Accuracy':>12}")
print("=" * 50)
for model_name, acc in results.items():
    print(f"{model_name:<35} {acc*100 :>12.2f}%")
print("=" * 50)

In [ ]:
# Compare train vs validation accuracy for image models (50 epochs)
plt.figure(figsize=(11, 6))

histories_50 = [
    (Net1_history_50, "Net1"),
    (Net2_history_50, "Net2"),
    (Net3_history_50, "Net3"),
    (Net4_history_50, "Net4")
]

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for (hist, name), color in zip(histories_50, colors):
    epochs = range(1, len(hist["val_acc"]) + 1)
    plt.plot(epochs, hist["train_acc"], color=color, linestyle="-",  label=f"{name} - Train")
    plt.plot(epochs, hist["val_acc"],   color=color, linestyle="--", label=f"{name} - Val")

plt.title("Image Models Comparison - 50 Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc=(0.7,0.7), fontsize=9,ncol=2)
plt.grid(True)
plt.tight_layout()
plt.savefig("results/comparison_50epochs.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Compare train vs validation accuracy for image models (100 epochs)
plt.figure(figsize=(11, 6))

histories_100 = [
    (Net1_history_100, "Net1"),
    (Net2_history_100, "Net2"),
    (Net3_history_100, "Net3"),
    (Net4_history_100, "Net4")
]

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for (hist, name), color in zip(histories_100, colors):
    epochs = range(1, len(hist["val_acc"]) + 1)
    plt.plot(epochs, hist["train_acc"], color=color, linestyle="-",  label=f"{name} - Train")
    plt.plot(epochs, hist["val_acc"],   color=color, linestyle="--", label=f"{name} - Val")

plt.title("Image Models Comparison - 100 Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc=(0.7,0.05), fontsize=9.5,ncol=2)
plt.grid(True)
plt.tight_layout()
plt.savefig("results/comparison_100epochs.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Compare train vs validation accuracy for image models (100 epochs)
plt.figure(figsize=(11, 6))

histories_100 = [
    (Net5_history, "Net5"),
    (Net6_history, "Net6"),
]

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for (hist, name), color in zip(histories_100, colors):
    epochs = range(1, len(hist["val_acc"]) + 1)
    plt.plot(epochs, hist["train_acc"], color=color, linestyle="-",  label=f"{name} - Train")
    plt.plot(epochs, hist["val_acc"],   color=color, linestyle="--", label=f"{name} - Val")

plt.title("Image Models Comparison - Net5 and Net6")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(loc=(0.7,0.05), fontsize=9,ncol=2)
plt.grid(True)
plt.tight_layout()
plt.savefig("results/comparison_net5_net6.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Grouped bar chart: 50 vs 100 epochs for Net1-Net4, plus Net5 and Net6
fig, ax = plt.subplots(figsize=(12, 6))

# Net1-Net4 accuracies (50 and 100 epochs)
models_img = ["Net1 (FC)", "Net2 (CNN)", "Net3 (CNN+BN)", "Net4 (CNN+BN+RMSprop)"]
acc_50  = [Net1_acc_50, Net2_acc_50, Net3_acc_50, Net4_acc_50]
acc_100 = [Net1_acc_100, Net2_acc_100, Net3_acc_100, Net4_acc_100]

# Audio models
models_audio = ["Net5(LSTM)", "Net6 (LSTM+GAN)"]
acc_audio = [Net5_acc, Net6_acc]

# --- Image models: grouped bars ---
x_img = np.arange(len(models_img))
bar_w = 0.35

bars_50  = ax.bar(x_img - bar_w/2, acc_50,  bar_w, label="50 Epochs",  color="#3498db")
bars_100 = ax.bar(x_img + bar_w/2, acc_100, bar_w, label="100 Epochs", color="#e74c3c")

# --- Audio models: single bars offset to the right ---
x_audio = np.arange(len(models_audio)) + len(models_img) + 0.5
bars_audio = ax.bar(x_audio, acc_audio, bar_w, label="Audio Models", color="#2ecc71")

# Add accuracy labels on every bar
for bars in [bars_50, bars_100, bars_audio]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f"{height:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

# X-axis
all_x = np.concatenate([x_img, x_audio])
all_labels = models_img + models_audio
ax.set_xticks(all_x)
ax.set_xticklabels(all_labels, fontsize=10)

# Divider line between image and audio models
ax.axvline(x=len(models_img) - 0.25, color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax.text(len(models_img) - 0.55, 0.95, "Image Models", ha="right", fontsize=9, fontstyle="italic", color="gray", transform=ax.get_xaxis_transform())
ax.text(len(models_img) + 0.05, 0.95, "Audio Models", ha="left", fontsize=9, fontstyle="italic", color="gray", transform=ax.get_xaxis_transform())

# Labels and styling
ax.set_ylabel("Test Accuracy", fontsize=12)
ax.set_title("Music Genre Classification — Model Accuracy Comparison", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(loc="upper left", fontsize=10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("results/model_comparison.png", dpi=200, bbox_inches="tight")
plt.show()